# 01_data_generation.ipynb
## Synthetic Dataset Generation for Personalized BKT in Mild Autism (Ages 3–10)

*Authors:* WASSWA COSMAS MUYOMBA 
*Date:* March 2026  
*Project:* Modified Bayesian Knowledge Tracing (BKT) Mobile App for Autistic Children

### Objective
Generate high-fidelity synthetic longitudinal data that mirrors documented learning patterns in children with mild Autism Spectrum Disorder (ASD). The simulator explicitly incorporates *higher forgetting, **variable behavior/engagement, and **increased hint reliance* — features repeatedly observed in ASD math learning research (Tonizzi et al., 2023; Polo-Blanco et al., 2022; Bejarano-Martín et al., 2024).

*Key literature grounding*:
- ASD children show poorer arithmetic performance and higher variability than typically developing (TD) peers (Tonizzi et al., 2023 meta-analysis).
- Mastery rates in adaptive learning for ASD are lower for harder skills (0.78 easy vs. <0.55 hard) with high guessing sensitivity (Dharsika et al., 2026 BKT-ADL study on 27 ASD learners).
- Forgetting is a critical extension for realistic long-term modeling (Lee et al., 2023; Khajah et al., 2016).

*Skills modeled* (your app modules):
- counting, shape_recognition, addition, subtraction, multiplication, division

Data saved to data/raw/ for downstream notebooks.

In [ ]:
# ================================================
# Imports & Setup
# ================================================
# Numerical operations and random sampling
import numpy as np
# Tabular data handling for generated records
import pandas as pd
# Cross-platform path handling for data folders/files
from pathlib import Path

# Reproducibility: fix RNG seed so repeated runs produce the same synthetic dataset
np.random.seed(42)

# Project paths: define where raw generated CSV files will be saved
DATA_RAW = Path("../data/raw")
# Ensure the output directory exists before writing files
DATA_RAW.mkdir(parents=True, exist_ok=True)

# Quick checkpoint message for notebook progress tracking
print("Setup complete")

Setup complete


In [ ]:
# ================================================
# Autism-Tuned Synthetic BKT Simulator
# ================================================
def generate_synthetic_bkt_data(
    n_students: int = 1200,
    n_opportunities_per_skill: int = 120,
    skills: list = ["counting", "shape_recognition", "addition", "subtraction", "multiplication", "division"],
    autism_mode: bool = True,
    output_path: str = None
) -> pd.DataFrame:
    """
    Simulator tuned for mild autism (ages 3-10).
    References: Tonizzi et al. (2023), Dharsika et al. (2026), Lee et al. (2023)
    """
    # Collect one interaction record per student-skill-opportunity
    data = []
    
    # Stage 1: iterate through students to assign learner-specific priors
    for student_id in range(n_students):
        # Initial knowledge prior (p_l0): autism group has lower mean and higher spread
        p_l0 = np.clip(np.random.normal(0.45 if autism_mode else 0.55, 0.18), 0.05, 0.95)
        # Base learning transition (p_t) before skill difficulty and behavior modulation
        base_p_t = np.random.uniform(0.08, 0.22) if autism_mode else np.random.uniform(0.15, 0.35)
        # Forgetting probability (p_f): intentionally higher in autism simulation profile
        p_f = np.random.uniform(0.06, 0.18) if autism_mode else np.random.uniform(0.01, 0.05)  # Your forgetting param
        # Guess probability (p_g): chance of correct response while unlearned
        p_g = np.random.uniform(0.18, 0.32) if autism_mode else np.random.uniform(0.10, 0.25)
        # Slip probability (p_s): chance of incorrect response while learned
        p_s = np.random.uniform(0.18, 0.32) if autism_mode else np.random.uniform(0.10, 0.25)
        # Baseline behavior/engagement center for this learner
        behavior_mean = np.random.uniform(0.55, 0.78) if autism_mode else np.random.uniform(0.75, 0.92)
        
        # Stage 2: iterate through each skill for the current student
        for skill in skills:
            # Skill-level difficulty modifier: later arithmetic skills learn slower
            difficulty_factor = 1.0 if skill in ["counting", "shape_recognition", "addition"] else 0.65
            # Skill-adjusted transition probability
            p_t = base_p_t * difficulty_factor
            
            # Initialize latent knowledge state at first opportunity using p_l0
            knowledge = 1 if np.random.rand() < p_l0 else 0
            
            # Stage 3: simulate repeated practice opportunities for this student-skill pair
            for opp in range(n_opportunities_per_skill):
                # Sample momentary behavior to emulate attention/engagement fluctuations
                behavior = np.clip(np.random.normal(behavior_mean, 0.22), 0.0, 1.0)
                
                # Hint policy: increase hint usage when learner is unlearned or disengaged
                hint_prob = 0.45 if (knowledge == 0 or behavior < 0.5) else 0.12
                hint_used = 1 if np.random.rand() < hint_prob else 0
                
                # Stage 4: adapt BKT parameters by behavior and hint usage for this step
                effective_p_t = p_t * (1 + 0.35 * hint_used) * behavior
                effective_p_s = p_s * (1 - 0.25 * behavior)
                
                # Stage 5: generate observed correctness from latent knowledge state
                if knowledge == 1:
                    # Learned state: mostly correct, except slip events
                    correct = 1 - np.random.binomial(1, effective_p_s)
                else:
                    # Unlearned state: occasional correct response via guessing
                    correct = np.random.binomial(1, p_g)
                
                # Stage 6: store one fully observed interaction row
                data.append({
                    "anon_student_id": student_id,
                    "skill_name": skill,
                    "correct": int(correct),
                    "hint_used": hint_used,
                    "behavior_score": round(behavior, 3),
                    "opportunity": opp + 1
                })
                
                # Stage 7: evolve latent knowledge for next opportunity
                if knowledge == 1:
                    # Learned state can decay via forgetting
                    knowledge = 0 if np.random.rand() < p_f else 1
                else:
                    # Unlearned state can transition to learned via effective learning probability
                    if np.random.rand() < effective_p_t:
                        knowledge = 1
    
    # Convert accumulated interaction records into a dataframe
    df = pd.DataFrame(data)
    # Optionally persist to CSV for downstream notebooks/pipelines
    if output_path:
        df.to_csv(output_path, index=False)
        print(f"✅ Saved {len(df):,} rows to {output_path}")
    # Return generated dataset to caller
    return df

# ================================================
# Generate BOTH datasets
# ================================================
# Autism-profile dataset used as primary training/evaluation corpus
df_autism = generate_synthetic_bkt_data(
    n_students=1200,
    autism_mode=True,
    output_path=DATA_RAW / "synthetic_autism_data.csv"
)

# Typical-profile comparison dataset for validation/benchmarking
df_typical = generate_synthetic_bkt_data(
    n_students=800,          # smaller for comparison
    autism_mode=False,
    output_path=DATA_RAW / "synthetic_typical_data.csv"
)

# Final checkpoint message with resulting autism dataset dimensions
print("Generation complete. Autism dataset shape:", df_autism.shape)

✅ Saved 864,000 rows to ..\data\raw\synthetic_autism_data.csv
✅ Saved 576,000 rows to ..\data\raw\synthetic_typical_data.csv
Generation complete. Autism dataset shape: (864000, 6)


### Summary Statistics (Autism vs Typical)
Run the cell below to preview — these will be used in our Notebook 02 for statistical validation.

In [ ]:
# Preview heading for quick visual separation in notebook output
print("Autism dataset sample:")
# Group by skill and summarize mean correctness, hint use, and behavior scores
display(df_autism.groupby("skill_name")[["correct", "hint_used", "behavior_score"]].mean().round(3))

Autism dataset sample:


,correct,hint_used,behavior_score
skill_name,,,
addition,0.513,0.326,0.660
counting,0.519,0.322,0.660
division,0.464,0.350,0.660
multiplication,0.464,0.348,0.661
shape_recognition,0.515,0.324,0.659
subtraction,0.463,0.350,0.660
